# Phase M5 — Video B v2: Fixed DDPM for Segmentation Interpolation
## 6 Fixes: Soft Labels | FiLM Conditioning | Hybrid Loss | 128×128 | 20M params | 50 steps
**Input:** `mu_glioma_triplets.npz` (128×128, downsampled to 128×128 internally)
**Output:** Trained DDPM → `/kaggle/working/video_train/ddpm_v2_best.pth`
---

In [ ]:
import os,time,math,gc,warnings
import numpy as np
import torch,torch.nn as nn,torch.nn.functional as F
from torch.utils.data import Dataset,DataLoader
from torch.cuda.amp import GradScaler,autocast
from pathlib import Path
warnings.filterwarnings('ignore')
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
CFG={'res':128,'n_cls':5,'bs':8,'T':50,'lr':2e-4,'epochs':300,
     'patience':35,'val_split':0.15,'base_ch':96}
OUT=Path('/kaggle/working/video_train');OUT.mkdir(parents=True,exist_ok=True)

# Set True to skip training and go straight to sampling/visualization
SKIP_TRAIN = False
print(f'Config: {CFG}')

In [ ]:
# Load (keep at 128x128 — NO downsampling)
tp=None
for p in Path('/kaggle/input').rglob('mu_glioma_triplets.npz'):tp=p;break
if tp is None:raise RuntimeError('triplets not found!')
d=np.load(tp)
seg_s,seg_e,seg_g=d['seg_start'],d['seg_end'],d['seg_gt']
t1c_s,t_interp,types=d['t1c_start'],d['t_interp'],d['types']
N=len(seg_s)
print(f'Loaded {N} samples at {seg_s.shape[1]}x{seg_s.shape[2]} (native resolution)')

In [ ]:
# Dataset with SOFT labels [-0.95, +0.95]
def seg_to_soft(seg,C=5):
    oh=np.zeros((C,seg.shape[0],seg.shape[1]),dtype=np.float32)
    for c in range(C):oh[c]=(seg==c).astype(np.float32)
    return oh*1.9-0.95  # map 0→-0.95, 1→+0.95

class DS(Dataset):
    def __init__(s,ss,se,sg,ti,t1c,aug=True):
        s.ss,s.se,s.sg,s.ti,s.t1c,s.aug=ss,se,sg,ti,t1c,aug
    def __len__(s):return len(s.ss)
    def __getitem__(s,i):
        ss=seg_to_soft(s.ss[i]);se=seg_to_soft(s.se[i])
        sg=seg_to_soft(s.sg[i]);gl=s.sg[i].astype(np.int64)
        t=np.float32(s.ti[i]);t1c=s.t1c[i].copy()
        if s.aug:
            if np.random.rand()>0.5:
                ss=ss[:,:,::-1].copy();se=se[:,:,::-1].copy()
                sg=sg[:,:,::-1].copy();gl=gl[:,::-1].copy();t1c=t1c[:,::-1].copy()
            if np.random.rand()>0.5:
                ss=ss[:,::-1].copy();se=se[:,::-1].copy()
                sg=sg[:,::-1].copy();gl=gl[::-1].copy();t1c=t1c[::-1].copy()
        return {'cond':torch.from_numpy(np.concatenate([ss,se],0)),
                'x0':torch.from_numpy(sg),'gt_labels':torch.from_numpy(gl),
                't_interp':torch.tensor([t]),'t1c':torch.from_numpy(t1c)}

np.random.seed(42);idx=np.random.permutation(N);nv=int(N*CFG['val_split'])
tr=DS(seg_s[idx[nv:]],seg_e[idx[nv:]],seg_g[idx[nv:]],t_interp[idx[nv:]],t1c_s[idx[nv:]],True)
va=DS(seg_s[idx[:nv]],seg_e[idx[:nv]],seg_g[idx[:nv]],t_interp[idx[:nv]],t1c_s[idx[:nv]],False)
tl=DataLoader(tr,batch_size=CFG['bs'],shuffle=True,num_workers=2,pin_memory=True,drop_last=True)
vl=DataLoader(va,batch_size=CFG['bs'],shuffle=False,num_workers=2,pin_memory=True)
print(f'Train:{len(tr)} Val:{len(va)} Batches:{len(tl)}/ep')

In [ ]:
# Cosine noise schedule T=50
def cosine_betas(T,s=0.008):
    t=torch.linspace(0,T,T+1)
    ac=torch.cos(((t/T)+s)/(1+s)*math.pi*0.5)**2
    ac=ac/ac[0];b=1-(ac[1:]/ac[:-1])
    return torch.clip(b,0.0001,0.9999)
T=CFG['T'];betas=cosine_betas(T).to(device)
alphas=1-betas;ac=torch.cumprod(alphas,0)
sac=torch.sqrt(ac);s1mac=torch.sqrt(1-ac)
acp=F.pad(ac[:-1],(1,0),value=1.0)
pv=betas*(1-acp)/(1-ac)
def q_sample(x0,t,noise=None):
    if noise is None:noise=torch.randn_like(x0)
    return sac[t][:,None,None,None]*x0+s1mac[t][:,None,None,None]*noise,noise
print(f'Noise schedule: cosine, T={T}')

In [ ]:
# FiLM-Conditioned U-Net (~20M params)
class SinPE(nn.Module):
    def __init__(s,d):super().__init__();s.d=d
    def forward(s,t):
        h=s.d//2;e=math.log(10000)/(h-1)
        e=torch.exp(torch.arange(h,device=t.device)*-e)
        e=t[:,None].float()*e[None,:];return torch.cat([e.sin(),e.cos()],-1)

class FiLMResBlock(nn.Module):
    def __init__(s,ic,oc,cond_ch,td):
        super().__init__()
        g1=min(8,ic) if ic%8==0 else(4 if ic%4==0 else 1)
        g2=min(8,oc) if oc%8==0 else(4 if oc%4==0 else 1)
        s.c1=nn.Sequential(nn.GroupNorm(g1,ic),nn.SiLU(),nn.Conv2d(ic,oc,3,1,1))
        s.c2=nn.Sequential(nn.GroupNorm(g2,oc),nn.SiLU(),nn.Conv2d(oc,oc,3,1,1))
        s.tp=nn.Sequential(nn.SiLU(),nn.Linear(td,oc))
        s.film=nn.Conv2d(cond_ch,oc*2,1)  # scale+shift from condition
        s.sk=nn.Conv2d(ic,oc,1) if ic!=oc else nn.Identity()
    def forward(s,x,te,cf):
        h=s.c1(x);h=h+s.tp(te)[:,:,None,None]
        # FiLM modulation
        fs=s.film(cf);sc,sh=fs.chunk(2,dim=1)
        h=h*(1+sc)+sh
        h=s.c2(h);return h+s.sk(x)

class CondEncoder(nn.Module):
    """Encode seg_start+seg_end into multi-scale features for FiLM."""
    def __init__(s,ic=10,base=96):
        super().__init__()
        s.e1=nn.Sequential(nn.Conv2d(ic,base,3,1,1),nn.SiLU())
        s.e2=nn.Sequential(nn.Conv2d(base,base*2,3,2,1),nn.SiLU())
        s.e3=nn.Sequential(nn.Conv2d(base*2,base*4,3,2,1),nn.SiLU())
        s.e4=nn.Sequential(nn.Conv2d(base*4,base*8,3,2,1),nn.SiLU())
    def forward(s,x):
        f1=s.e1(x);f2=s.e2(f1);f3=s.e3(f2);f4=s.e4(f3)
        return [f1,f2,f3,f4]

class DDPMv2(nn.Module):
    def __init__(s,x_ch=5,cond_ch=10,B=96,td=256):
        super().__init__()
        s.t_emb=nn.Sequential(SinPE(td),nn.Linear(td,td),nn.SiLU())
        s.ti_emb=nn.Sequential(nn.Linear(1,td),nn.SiLU(),nn.Linear(td,td))
        s.cond_enc=CondEncoder(cond_ch,B)
        s.inp=nn.Sequential(nn.Conv2d(x_ch,B,3,1,1),nn.SiLU())
        # Encoder: FiLM at each level
        s.e1=FiLMResBlock(B,B,B,td)
        s.e2=FiLMResBlock(B,B*2,B*2,td)
        s.e3=FiLMResBlock(B*2,B*4,B*4,td)
        s.e4=FiLMResBlock(B*4,B*8,B*8,td)
        s.dn=nn.MaxPool2d(2)
        s.up=nn.Upsample(scale_factor=2,mode='bilinear',align_corners=False)
        s.d3=FiLMResBlock(B*8+B*4,B*4,B*4,td)
        s.d2=FiLMResBlock(B*4+B*2,B*2,B*2,td)
        s.d1=FiLMResBlock(B*2+B,B,B,td)
        s.out=nn.Conv2d(B,x_ch,1)
    def forward(s,xn,td_,cond,ti):
        te=s.t_emb(td_)+s.ti_emb(ti)
        cf=s.cond_enc(cond)  # [f1_64,f2_32,f3_16,f4_8]
        x=s.inp(xn)
        e1=s.e1(x,te,cf[0])
        e2=s.e2(s.dn(e1),te,cf[1])
        e3=s.e3(s.dn(e2),te,cf[2])
        e4=s.e4(s.dn(e3),te,cf[3])
        d3=s.d3(torch.cat([s.up(e4),e3],1),te,cf[2])
        d2=s.d2(torch.cat([s.up(d3),e2],1),te,cf[1])
        d1=s.d1(torch.cat([s.up(d2),e1],1),te,cf[0])
        return s.out(d1)

model=DDPMv2(x_ch=5,cond_ch=10,B=CFG['base_ch']).to(device)
np_=sum(p.numel() for p in model.parameters())/1e6
print(f'DDPMv2: {np_:.1f}M params')
# Shape test
with torch.no_grad():
    xn=torch.randn(2,5,64,64).to(device)
    cd=torch.randn(2,10,64,64).to(device)
    tt=torch.randint(0,T,(2,)).to(device)
    ti=torch.rand(2,1).to(device)
    o=model(xn,tt,cd,ti);print(f'Shape: {o.shape} ok')
    del xn,cd,o;torch.cuda.empty_cache()

In [ ]:
# Hybrid Loss: weighted MSE + Dice on x0_pred + focal CE
CW=torch.tensor([0.1,2.0,2.0,0.1,3.0],device=device)

def predict_x0(xt,t,noise_pred):
    return(xt-s1mac[t][:,None,None,None]*noise_pred)/sac[t][:,None,None,None].clamp(min=1e-5)

def soft_dice(logits,target,smooth=1.0):
    p=torch.softmax(logits,1)
    toh=F.one_hot(target,5).permute(0,3,1,2).float()
    inter=(p*toh).sum((2,3));union=p.sum((2,3))+toh.sum((2,3))
    d=(2*inter+smooth)/(union+smooth)
    return 1-d[:,1:].mean()  # skip bg

def hybrid_loss(noise_pred,noise,xt,t,gt_labels,x0_soft):
    # 1. Spatially weighted MSE (tumor 50x)
    tmask=(gt_labels>0).float().unsqueeze(1).expand_as(noise)
    w=1.0+49.0*tmask
    mse=(w*(noise_pred-noise)**2).mean()
    # 2. Dice on predicted x0
    x0p=predict_x0(xt,t,noise_pred)
    dl=soft_dice(x0p,gt_labels)
    # 3. Focal CE on predicted x0
    ce=F.cross_entropy(x0p,gt_labels,weight=CW)
    return mse+0.5*dl+0.3*ce

def dice_sc(pred,gt):
    r={}
    for n,pm,gm in[('WT',pred>0,gt>0),('TC',(pred==1)|(pred==4),(gt==1)|(gt==4)),('ET',pred==4,gt==4)]:
        inter=(pm&gm).sum().float();union=pm.sum().float()+gm.sum().float()
        r[n]=(2*inter/(union+1e-8)).item()
    return r
print('Hybrid loss ready (wMSE + Dice + focalCE)')

In [ ]:
# ── Resume / Skip Training ─────────────────────────────────────
import shutil
CKPT = OUT / 'ddpm_v2_128_best.pth'
se = 0; best_dice = 0.0; tl_ = []; vl_ = []; vd_ = []

ckpt_src = None
for p in sorted(Path('/kaggle/input').rglob('ddpm_v2_128_best.pth')):
    ckpt_src = p; break

if ckpt_src:
    ck = torch.load(ckpt_src, map_location=device, weights_only=False)
    model.load_state_dict(ck['model'])
    se = ck.get('epoch', 0) + 1
    best_dice = ck.get('best_dice', 0.0)
    shutil.copy(str(ckpt_src), str(CKPT))
    print(f'✅ Checkpoint loaded: ep{se} | best_dice={best_dice:.4f}')
else:
    print('No checkpoint found — starting from scratch')

if SKIP_TRAIN:
    print('⏭  SKIP_TRAIN=True — skipping training, proceeding to sampling')
else:
    # ── Training ────────────────────────────────────────────────
    def dice_sc(p, g):
        r = {}
        for n, pm, gm in [('WT', p>0, g>0), ('TC', (p==1)|(p==4), (g==1)|(g==4)), ('ET', p==4, g==4)]:
            r[n] = (2*(pm&gm).sum().float()/(pm.sum().float()+gm.sum().float()+1e-8)).item()
        return r

    opt = torch.optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['epochs'], eta_min=1e-6)
    pat = 0
    print(f'Training ep{se}→{CFG["epochs"]} | patience={CFG["patience"]} | FP32')
    t0 = time.time()

    for ep in range(se, CFG['epochs']):
        model.train(); el = []
        for b in tl:
            cond=b['cond'].to(device); x0=b['x0'].to(device)
            gl=b['gt_labels'].to(device); ti=b['t_interp'].to(device)
            B_=cond.shape[0]; t=torch.randint(0,T,(B_,),device=device)
            xt,noise=q_sample(x0,t); np_=model(xt,t,cond,ti)
            loss=hybrid_loss(np_,noise,xt,t,gl,x0)
            if torch.isnan(loss) or torch.isinf(loss): continue
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 0.5)
            opt.step(); el.append(loss.item())

        sch.step()
        if not el: print(f'  Ep{ep+1}: all NaN, skipping'); continue
        tl_.append(np.mean(el))

        model.eval(); vll=[]; vd={'WT':[],'TC':[],'ET':[]}
        with torch.no_grad():
            for b in vl:
                cond=b['cond'].to(device); x0=b['x0'].to(device)
                gl=b['gt_labels'].to(device); ti=b['t_interp'].to(device)
                B_=cond.shape[0]; t=torch.randint(0,T,(B_,),device=device)
                xt,noise=q_sample(x0,t); np_=model(xt,t,cond,ti)
                if not torch.isnan(np_).any():
                    vll.append(hybrid_loss(np_,noise,xt,t,gl,x0).item())
                    pl=predict_x0(xt,t,np_).argmax(1)
                    for i in range(B_):
                        ds=dice_sc(pl[i],gl[i])
                        for k in ds: vd[k].append(ds[k])

        if not vll: continue
        vl_.append(np.mean(vll))
        md_ = np.mean([np.mean(vd[k]) for k in vd]); vd_.append(md_)
        imp = md_ > best_dice
        if imp:
            best_dice=md_; pat=0
            torch.save({'model':model.state_dict(),'epoch':ep,'best_dice':best_dice,'cfg':CFG}, str(CKPT))
        else:
            pat += 1

        if (ep+1)%5==0 or imp:
            print(f'  Ep{ep+1:3d}/{CFG["epochs"]} l={tl_[-1]:.4f}/{vl_[-1]:.4f} '
                  f'WT={np.mean(vd["WT"]):.3f} TC={np.mean(vd["TC"]):.3f} ET={np.mean(vd["ET"]):.3f} '
                  f'avg={md_:.3f} best={best_dice:.3f} pat={pat} {time.time()-t0:.0f}s')
        if pat >= CFG['patience']: print(f'  Early stop ep{ep+1}'); break

    print(f'Done. Best Dice: {best_dice:.4f}')

In [ ]:
# Training curves (skipped if SKIP_TRAIN=True)
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
if tl_ and vl_:
    fig,(a1,a2)=plt.subplots(1,2,figsize=(14,5))
    a1.plot(tl_,label='Train',color='#1565C0'); a1.plot(vl_,label='Val',color='#E53935')
    a1.set_xlabel('Epoch'); a1.set_ylabel('Hybrid Loss'); a1.legend(); a1.grid(True,alpha=0.3)
    a1.set_title('Loss Curves')
    a2.plot(vd_,label='Avg Dice',color='#2E7D32',lw=2)
    a2.axhline(best_dice,ls='--',color='gray',label=f'Best:{best_dice:.3f}')
    a2.set_xlabel('Epoch'); a2.set_ylabel('Dice'); a2.legend(); a2.grid(True,alpha=0.3)
    a2.set_title('Val Dice')
    plt.tight_layout(); plt.savefig(str(OUT/'curves_v2.png'),dpi=150); plt.show()
    print('Curves saved')
else:
    print('No training history to plot (SKIP_TRAIN=True or checkpoint loaded)')

In [ ]:
# Full 50-step DDPM sampling + Dice validation
ck=torch.load(str(CKPT),map_location=device,weights_only=False)
model.load_state_dict(ck['model']);model.eval()

@torch.no_grad()
def sample_full(model,cond,ti_val):
    "Full 50-step DDPM reverse from pure noise"
    B=cond.shape[0]
    x=torch.randn(B,5,128,128,device=device)
    ti=torch.full((B,1),ti_val,device=device)
    for i in reversed(range(T)):
        t=torch.full((B,),i,device=device,dtype=torch.long)
        np_=model(x,t,cond,ti)
        a=alphas[i];ab=ac[i];bt=betas[i]
        x0_pred=(x-torch.sqrt(1-ab)*np_)/torch.sqrt(ab).clamp(min=1e-5)
        x0_pred=x0_pred.clamp(-2,2)
        mean=(1/torch.sqrt(a))*(x-(bt/torch.sqrt(1-ab))*np_)
        if i>0:x=mean+torch.sqrt(pv[i])*torch.randn_like(x)
        else:x=mean
    return x

def label_rgb(s):
    r=np.zeros((*s.shape,3),dtype=np.float32)
    r[s==1]=[0.2,0.4,1.0];r[s==2]=[0.2,0.8,0.3];r[s==4]=[1.0,0.2,0.2];return r
def overlay(seg,t1c,a=0.6):
    tr=np.stack([t1c]*3,-1);sr=label_rgb(seg);m=sr.sum(-1,keepdims=True)>0
    return np.clip(np.where(m,(1-a)*tr+a*sr,tr),0,1)
def enforce_containment(seg):
    out=seg.clone();tc=(out==1)|(out==4);et=out==4
    out[et&~tc]=1;return out

# Validate on 30 val samples with full sampling
print('Running full 50-step validation on 30 samples...')
vd_full={'WT':[],'TC':[],'ET':[]}
for i in range(min(30,len(va))):
    s=va[i];cond=s['cond'].unsqueeze(0).to(device)
    ti=s['t_interp'].item();gt=s['gt_labels'].numpy()
    pred_raw=sample_full(model,cond,ti).argmax(1)[0]
    pred=enforce_containment(pred_raw).cpu().numpy()
    d=dice_sc(torch.tensor(pred),torch.tensor(gt))
    for k in d:vd_full[k].append(d[k])
print(f'Full 50-step Dice (30 samples):')
print(f'  WT={np.mean(vd_full["WT"]):.3f} TC={np.mean(vd_full["TC"]):.3f} ET={np.mean(vd_full["ET"]):.3f} avg={np.mean([np.mean(vd_full[k]) for k in vd_full]):.3f}')
print(f'  (Previous 25-step: WT~0.953 TC~0.724 ET~0.601)')

# Visual test — 4 samples
import matplotlib;matplotlib.use('Agg');import matplotlib.pyplot as plt
fig,axes=plt.subplots(4,4,figsize=(16,16))
fig.suptitle('DDPMv2 128x128 — Full 50-step Sampling on Brain MRI',fontsize=13,fontweight='bold')
for j,c in enumerate(['Start','GT','Predicted (50-step)','End']):axes[0,j].set_title(c,fontsize=11,fontweight='bold')
for row in range(4):
    s=va[row*5];cond=s['cond'].unsqueeze(0).to(device)
    ti=s['t_interp'].item();t1c=s['t1c'].numpy();gt=s['gt_labels'].numpy()
    pred_raw=sample_full(model,cond,ti).argmax(1)[0]
    pred=enforce_containment(pred_raw).cpu().numpy()
    sl=s['cond'][:5].argmax(0).numpy();el=s['cond'][5:].argmax(0).numpy()
    d=dice_sc(torch.tensor(pred),torch.tensor(gt))
    for j,seg in enumerate([sl,gt,pred,el]):
        axes[row,j].imshow(overlay(seg,t1c),origin='lower');axes[row,j].axis('off')
    axes[row,2].set_xlabel(f'WT={d["WT"]:.2f} TC={d["TC"]:.2f} ET={d["ET"]:.2f}',fontsize=9)
plt.tight_layout();plt.savefig(str(OUT/'sampling_50step.png'),dpi=150,bbox_inches='tight');plt.show()
print('Saved sampling_50step.png')

In [ ]:
# Interpolation sequence — full 50-step sampling
s=va[0];cond=s['cond'].unsqueeze(0).to(device);t1c=s['t1c'].numpy()
fig,axes=plt.subplots(1,11,figsize=(22,2.5))
fig.suptitle('Tumor Evolution (Full 50-step): t=0->1',fontsize=13,fontweight='bold')
for i,tv in enumerate(np.linspace(0,1,11)):
    pred_raw=sample_full(model,cond,tv).argmax(1)[0]
    pred=enforce_containment(pred_raw).cpu().numpy()
    axes[i].imshow(overlay(pred,t1c),origin='lower')
    axes[i].set_title(f't={tv:.1f}',fontsize=9);axes[i].axis('off')
plt.tight_layout();plt.savefig(str(OUT/'interp_50step.png'),dpi=150,bbox_inches='tight');plt.show()
print('Saved interp_50step.png')

In [ ]:
# Summary
print('\n'+'='*60)
print('  DDPMv2 128x128 — COMPLETE')
print('='*60)
n_p = sum(p.numel() for p in model.parameters())/1e6
print(f'  Model:    DDPMv2 ({n_p:.1f}M params)')
print(f'  Res:      {CFG["res"]}x{CFG["res"]}')
print(f'  T:        {T} steps (sampling: 25 warm-start)')
print(f'  Best Dice:{best_dice:.4f}')
print(f'  Epochs:   {len(tl_)} trained this session')
print(f'  Files:')
for f in sorted(OUT.iterdir()): print(f'    {f.name:35s} {f.stat().st_size/1e6:.1f}MB')
print(f'  → Next: Video_C_Validate.ipynb')
print('='*60)